In [ ]:
import json
from docx import Document
from docx.oxml.ns import qn
import zipfile
from lxml import etree

# Functions

In [ ]:
def extract_comments_part(doc, docx_path=None):
    """
    Return a dict mapping comment IDs (as strings) to comment text.
    Tries python-docx API first; if unavailable, parses word/comments.xml from the .docx ZIP.
    """
    comments_dict = {}
    # Try python-docx built-in part
    try:
        comments_part = doc.part.comments_part
        for comment in comments_part.comments:
            cid = comment._element.get(qn('w:id'))
            texts = [t.text for t in comment._element.xpath('.//w:t') if t.text]
            comments_dict[cid] = ''.join(texts)
        return comments_dict
    except Exception:
        pass
    # Fallback: direct ZIP parsing of word/comments.xml
    if docx_path is None:
        return comments_dict
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    try:
        with zipfile.ZipFile(docx_path) as z:
            data = z.read('word/comments.xml')
    except Exception:
        return comments_dict
    root = etree.fromstring(data)
    for comment in root.xpath('//w:comment', namespaces=ns):
        cid = comment.get('{%s}id' % ns['w'])
        texts = comment.xpath('.//w:t/text()', namespaces=ns)
        comments_dict[cid] = ''.join(texts)
    return comments_dict

def run_has_comment_reference(run):
    """
    If <w:commentReference> is a child of run._r, return the comment ID.
    Otherwise return None.
    """
    for child in run._r:
        if child.tag == qn("w:commentReference"):
            comment_id = child.get(qn("w:id"))
            return comment_id
    return None

def run_is_within_comment_range(run):
    """
    If <w:commentRangeStart> is a child of run._r, returns the comment ID.
    (This is a comment that wraps multiple runs.)
    Otherwise return None.
    """
    for child in run._r:
        if child.tag == qn("w:commentRangeStart"):
            comment_id = child.get(qn("w:id"))
            return comment_id
    return None

def run_is_comment_range_end(run):
    """
    If <w:commentRangeEnd> is a child of run._r, returns the comment ID.
    Otherwise return None.
    """
    for child in run._r:
        if child.tag == qn("w:commentRangeEnd"):
            return child.get(qn("w:id"))
    return None

def gather_paragraph_annotations(paragraph, comments_dict):
    """
    Return a list of { highlighted_text, comment, context_left, context_right }
    for this paragraph, combining highlights & comments.
    """
    full_text = paragraph.text
    annotations = []
    seen = set()

    # comment‐range blocks (wrap multiple runs)
    p_elem = paragraph._p
    children = list(p_elem)
    i = 0
    while i < len(children):
        child = children[i]
        if child.tag == qn("w:commentRangeStart"):
            cid = child.get(qn("w:id"))
            seen.add(cid)
            # collect all text up to matching End
            collected = []
            j = i + 1
            while j < len(children):
                c2 = children[j]
                if c2.tag == qn("w:commentRangeEnd") and c2.get(qn("w:id")) == cid:
                    break
                if c2.tag == qn("w:r"):
                    for t in c2.xpath(".//w:t"):
                        collected.append(t.text or "")
                j += 1
            ht = "".join(collected)
            if ht:
                start = full_text.find(ht)
                if start >= 0:
                    end = start + len(ht)
                    left = full_text[max(0, start-100):start]
                    right = full_text[end:end+100]
                else:
                    left = right = ""
                annotations.append({
                    "type": "highlight",
                    "color": "generic",
                    "text": ht,
                    "comment": comments_dict.get(cid, ""),
                    "context_left": left,
                    "context_right": right
                })
            i = j
        i += 1

    # per‐run annotations (leftover highlights or point comments)
    for run in paragraph.runs:
        cid_ref = run_has_comment_reference(run)
        cid_range = run_is_within_comment_range(run)
        cid = cid_ref or cid_range
        if cid and cid in seen:
            continue
        is_hl = bool(run.font.highlight_color)
        is_pt = bool(cid_ref and cid_ref in comments_dict)
        if not (is_hl or is_pt):
            continue
        ht = run.text
        if not ht:
            continue
        start = full_text.find(ht)
        if start < 0:
            continue
        end = start + len(ht)
        left = full_text[max(0, start-100):start]
        right = full_text[end:end+100]
        annotations.append({
            "type": "highlight",
            "color": "generic",
            "highlighted_text": ht,
            "comment": comments_dict.get(cid, ""),
            "context_left": left,
            "context_right": right
        })
        if cid:
            seen.add(cid)

    return annotations

def docx_to_json(docx_path):
    """
    Open the DOCX at docx_path, parse paragraph by paragraph,
    extract highlights & comments as structured JSON.
    Returns a dict { "paragraphs": [ { "body": "...", "annotations": [ ... ] }, ... ] }
    """
    doc = Document(docx_path)
    comments_dict = extract_comments_part(doc, docx_path)

    output = {"paragraphs": []}
    for para in doc.paragraphs:
        p_text = para.text
        if not p_text.strip():
            # include them with no annotations
            output["paragraphs"].append({
                "body": "",
                "annotations": []
            })
            continue

        annots = gather_paragraph_annotations(para, comments_dict)
        output["paragraphs"].append({
            "body": p_text,
            "annotations": annots
        })

    return output


# Run

In [ ]:
docx_file = "... .docx"

json_file = "output.json"
result = docx_to_json(docx_file)

#print(result)
with open(json_file, "w", encoding="utf-8") as f_out:
    json.dump(result, f_out, ensure_ascii=False, indent=2)
